# Getting Started with Opentrons (OT-2) — Jupyter Notebook Guide

### Guide made by Lucas Levassor 

This short guide gives you a hands-on introduction to the Opentrons OT-2 liquid-handling robot.
By the end, you’ll understand what it is, what it can (and can’t) do, and how to run your first simple protocol.

## 1. What you’ll learn

- What the Opentrons OT-2 is and its main limitations

- How to understand the deck, pipettes, and labware

- How to use the robot step by step

- How to write and simulate a minimal Python protocol

- Common pitfalls and how to avoid them (tips, heights, volumes, speeds)

## 2. What is an Opentrons OT-2?

The Opentrons OT-2 is a bench-top pipetting robot that runs open Python scripts to perform liquid-handling steps such as:

- Transferring liquids between wells and tubes

- Mixing samples

- Preparing plates for assays or PCR

- It works by reading a Python protocol that tells it which labware is on which deck slot, which pipette to use, and what volumes to move.

#### Advantages

- Easy to use and programmable in Python

- Open-source and affordable

- Good for reproducible liquid-handling workflows

#### Limitations

- Only does liquid handling (no shaking, centrifugation, heating, or sensing) - although some extra modules can be added to support heating and shaking. 

- Limited precision for very small volumes (<2 µL)

- No feedback — if something spills, it doesn’t know

- Slow compared to expensive robots (but fine for most assays)

# 3. Understanding the deck, pipettes, and labware

- The deck

The OT-2 deck has 12 slots, numbered like this:


![OT-2 deck layout](Opentron_layout.png)


#### Pipettes

You can mount one or two pipettes:

- Single-channel for individual wells (manual: https://opentrons-landing-img.s3.amazonaws.com/pipettes/Opentrons-GEN2-Pipette-Single-Channel-Brochure-Digital.pdf)
- 8-channel (multi) for entire columns (manual: https://opentrons-landing-img.s3.amazonaws.com/pipettes/Opentrons-GEN2-Pipette-8-Channel-Brochure-Digital.pdf)

Each pipette uses specific tip racks (e.g., 20 µL, 300 µL, or 1000 µL).

And in the python-code we have to define which pipettes we use on which side they are mounted. Either 'left' or 'right'. For example: 



```python 
pipette = protocol.load_instrument("p300_multi_gen2", "right" tip_racks=tipracks) 
```

With that line of code we have loaded the p300 multi pipette on the right side and added the tipracks to it (more on this later)

#### Labware

Plates and reservoirs are defined by names in the API (e.g., "corning_96_wellplate_360ul_flat").
You tell the robot which slot each one occupies.

Here is a website where you can find what kind of labware they can work with and what they should be called in the code: https://labware.opentrons.com/






Example: 
```python 
reservoir = protocol.load_labware("nest_12_reservoir_15ml", reservoir_slot)
```

### Guides

They have cool guides here: https://support.opentrons.com/s/ot-2/get-started-guide 

# 4. How to actually use it - scripting

We start by adding importing the library and addning a metadata dict:

```python 
from opentrons import protocol_api

metadata = {"protocolName": "Hello OT-2", "apiLevel": "2.15"}
```


#### Then we can make a VERY SIMPLE Program that: 

1. Loads plates, resorvoir, tips and the p300 pipette. Notice that we after loading it have writtin the position on the deck
2. We pick up a tip
3. We tranfer 100 µl from the reservoir in position A1 to the plate in A1. 
3. We drop the tip (in position 12 the trash)

```python 
def run(protocol: protocol_api.ProtocolContext):
    plate = protocol.load_labware("corning_96_wellplate_360ul_flat", "2")
    reservoir = protocol.load_labware("nest_12_reservoir_15ml", "5")
    tips = protocol.load_labware("opentrons_96_tiprack_300ul", "8")
    p300 = protocol.load_instrument("p300_single", "right", tip_racks=[tips])

    p300.pick_up_tip()
    p300.transfer(100, reservoir["A1"], plate["A1"])
    p300.drop_tip()

```


## 5. Simulate before running in your terminal

```bash
opentrons_simulate first_protocol.py

```

Run the command above and see how the script is working

##  6. Common mistakes

| Issue                     | Why it happens                             | How to avoid it                                 |
| ------------------------- | ------------------------------------------ | ----------------------------------------------- |
| 💧 **Aspirating air**     | Tip too high above liquid                  | Use `well.bottom(z=1–2)` for aspiration         |
| ⚠️ **Crash into labware** | Wrong slot or uncalibrated plate           | Always simulate and perform labware calibration |
| 🧴 **Splashing**          | Flow rate too high                         | Lower `pip.flow_rate.dispense` to 50–80 µL/s    |
| 🧪 **Tip shortage**       | Forget to define enough racks              | Add multiple tip racks to `tip_racks=[...]`     |
| ⚗️ **Mixing unevenly**    | Not enough repetitions or too small volume | Use `pip.mix(3, 70% of max_volume)`             |


All of these can be fixed so it is a good idea to simulate or try your protocols before you do you do it the first time. 

## 7. How to actually run you script

1. What I do is that i test the script through the opentron app. https://opentrons.com/ot-app . Here you will see if you forgot to load enough tips etc. 


2. Our Opentron is not connected so to actually run a script i load it onto a thumbdrive and walk to the lab with it and transfer it to the opentron app. 

- When you have done that you can follow the instructions on screen and you should be good to go. 


## A word of advice: Play with it try it out and have fun. 


# 8. More information 

For more advanced stuff read the api guide - WHICH IS AMAZING! 

https://docs.opentrons.com/ot1/api.html#robot 


And here is a link to their github: https://github.com/Opentrons/opentrons/tree/edge


### THANK YOU FOR READING. More guides are coming
